# 📈 Stock Market Prediction — Multivariate LSTM Model

**Upgrades over v1:**
- ✅ Multivariate input: `Close + Volume + RSI + MACD` (4 features instead of 1)
- ✅ Extended training data: 2012 → 2024
- ✅ Lookback window increased: 100 → 150 days
- ✅ Per-feature scaling (each feature scaled independently)
- ✅ EarlyStopping + ModelCheckpoint retained
- ✅ MAE / RMSE / MAPE evaluation metrics
- ✅ **Indian stocks (NSE/BSE)** via Yahoo Finance — use `.NS` or `.BO` suffix

**Indian ticker examples (Yahoo Finance):**
| Company | NSE ticker | BSE ticker |
|---|---|---|
| Reliance | `RELIANCE.NS` | `RELIANCE.BO` |
| TCS | `TCS.NS` | `TCS.BO` |
| Infosys | `INFY.NS` | `INFY.BO` |
| HDFC Bank | `HDFCBANK.NS` | `HDFCBANK.BO` |

Set `MARKET = 'IN'` and pick a name from `INDIAN_STOCKS` in the config cell below.

**Sections:**
1. Imports & Config
2. Data Download & Cleaning
3. Feature Engineering (RSI, MACD, Volume)
4. Exploratory Analysis
5. Preprocessing & Scaling
6. Sequence Building
7. Model Architecture
8. Training
9. Evaluation
10. Save Model

## 1. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
from keras.layers import Dense, Dropout, LSTM, Input
from keras.models import Sequential
from keras.callbacks import EarlyStopping, ModelCheckpoint

import warnings
warnings.filterwarnings('ignore')

# ── Market & stock config ─────────────────────────────────
# MARKET: 'IN' = India (NSE/BSE), 'US' = United States
MARKET = 'IN'

# Pick a key from the dict, or set STOCK_NAME to a full Yahoo ticker (e.g. 'RELIANCE.NS')
INDIAN_STOCKS = {
    'RELIANCE':   'RELIANCE.NS',
    'TCS':        'TCS.NS',
    'INFY':       'INFY.NS',
    'HDFCBANK':   'HDFCBANK.NS',
    'ICICIBANK':  'ICICIBANK.NS',
    'SBIN':       'SBIN.NS',
    'BHARTIARTL': 'BHARTIARTL.NS',
    'ITC':        'ITC.NS',
    'WIPRO':      'WIPRO.NS',
    'TATAMOTORS': 'TATAMOTORS.NS',
    'NIFTY50':    '^NSEI',      # Nifty 50 index
}

US_STOCKS = {
    'GOOGLE': 'GOOG',
    'APPLE':  'AAPL',
    'MSFT':   'MSFT',
    'AMZN':   'AMZN',
    'TSLA':   'TSLA',
    'NVDA':   'NVDA',
}

STOCK_NAME = 'RELIANCE'   # change to any key above, or use full ticker


def resolve_ticker(symbol: str, market: str = MARKET) -> str:
    """Map friendly name → Yahoo Finance ticker (.NS / .BO for India)."""
    symbol = symbol.strip().upper()
    catalog = INDIAN_STOCKS if market.upper() == 'IN' else US_STOCKS
    if symbol in catalog:
        return catalog[symbol]
    # Already a Yahoo ticker (RELIANCE.NS, TCS.BO, AAPL, ^NSEI)
    if market.upper() == 'IN' and not (
        symbol.endswith('.NS') or symbol.endswith('.BO') or symbol.startswith('^')
    ):
        return f'{symbol}.NS'   # default to NSE if suffix omitted
    return symbol


STOCK       = resolve_ticker(STOCK_NAME, MARKET)
START       = '2012-01-01'
END         = '2025-12-31'
LOOKBACK    = 150
TRAIN_SPLIT = 0.80
FEATURES    = ['Close', 'Volume', 'RSI', 'MACD']
MODEL_PATH  = 'Stock Predictions Model.keras'

print(f"Market : {'India (NSE/BSE)' if MARKET == 'IN' else 'United States'}")
print(f"Ticker : {STOCK}")
# ────────────────────────────────────────────────────────

## 2. Data Download & Cleaning

Data is pulled from **Yahoo Finance**. Indian listings must use:
- **NSE:** `SYMBOL.NS` (e.g. `RELIANCE.NS`, `TCS.NS`)
- **BSE:** `SYMBOL.BO` (e.g. `RELIANCE.BO`)

If download returns 0 rows, check the ticker suffix or try another symbol.

In [ ]:
data = yf.download(STOCK, START, END, auto_adjust=True, progress=False)

# Flatten MultiIndex columns if present (yfinance ≥ 0.2.x)
if isinstance(data.columns, pd.MultiIndex):
    data.columns = data.columns.get_level_values(0)

if data.empty:
    examples = ', '.join(list(INDIAN_STOCKS.values())[:5]) if MARKET == 'IN' else ', '.join(list(US_STOCKS.values())[:5])
    raise ValueError(
        f"No data for '{STOCK}'. Check ticker and date range.\n"
        f"Indian stocks need .NS (NSE) or .BO (BSE), e.g. {examples}"
    )

# Drop NaN immediately — before any computation
data.dropna(inplace=True)

try:
    CURRENCY = yf.Ticker(STOCK).info.get('currency') or ('INR' if MARKET == 'IN' else 'USD')
except Exception:
    CURRENCY = 'INR' if MARKET == 'IN' else 'USD'

try:
    COMPANY = yf.Ticker(STOCK).info.get('longName', STOCK)
except Exception:
    COMPANY = STOCK

print(f"Company : {COMPANY}")
print(f"Market  : {'India (NSE/BSE)' if MARKET == 'IN' else 'United States'}")
print(f"Ticker  : {STOCK}")
print(f"Currency: {CURRENCY}")
print(f"Rows    : {len(data)}")
print(f"Range   : {data.index[0].date()} → {data.index[-1].date()}")
data[['Open','High','Low','Close','Volume']].tail()

## 3. Feature Engineering

We compute three technical indicators and add them as additional input channels:

| Feature | Why it helps |
|---|---|
| **Volume** | Confirms trend strength — high volume breakouts are more reliable |
| **RSI (14)** | Detects overbought (>70) / oversold (<30) — helps predict reversals |
| **MACD** | Momentum crossover signal — helps detect trend changes early |

In [ ]:
def compute_rsi(series, period=14):
    delta = series.diff()
    gain  = delta.clip(lower=0).rolling(period).mean()
    loss  = (-delta.clip(upper=0)).rolling(period).mean()
    rs    = gain / loss.replace(0, np.nan)
    return 100 - (100 / (1 + rs))

def compute_macd(series, fast=12, slow=26, signal=9):
    ema_fast   = series.ewm(span=fast,   adjust=False).mean()
    ema_slow   = series.ewm(span=slow,   adjust=False).mean()
    macd_line  = ema_fast - ema_slow
    signal_line = macd_line.ewm(span=signal, adjust=False).mean()
    return macd_line - signal_line   # MACD histogram (momentum)

close = data['Close'].squeeze()

data['RSI']  = compute_rsi(close)
data['MACD'] = compute_macd(close)

# Drop NaN rows created by rolling windows (first ~26 rows)
data.dropna(inplace=True)

print(f"Rows after feature engineering: {len(data)}")
data[FEATURES].tail()

## 4. Exploratory Analysis

In [ ]:
close = data['Close'].squeeze()
ma_50  = close.rolling(50).mean()
ma_100 = close.rolling(100).mean()
ma_200 = close.rolling(200).mean()

fig, axes = plt.subplots(4, 1, figsize=(14, 16), sharex=True)

# Price + MAs
axes[0].plot(data.index, close,  color='green', linewidth=1,   label='Close')
axes[0].plot(data.index, ma_50,  color='orange',linewidth=1.4, label='MA50')
axes[0].plot(data.index, ma_100, color='blue',  linewidth=1.4, label='MA100')
axes[0].plot(data.index, ma_200, color='red',   linewidth=1.4, label='MA200')
axes[0].set_title(f'{STOCK} — Close Price & Moving Averages')
axes[0].legend(); axes[0].grid(alpha=0.3)

# Volume
axes[1].bar(data.index, data['Volume'].squeeze(), color='steelblue', alpha=0.6, width=1)
axes[1].set_title('Volume')
axes[1].grid(alpha=0.3)

# RSI
axes[2].plot(data.index, data['RSI'].squeeze(), color='purple', linewidth=1)
axes[2].axhline(70, color='red',   linestyle='--', linewidth=1, label='Overbought (70)')
axes[2].axhline(30, color='green', linestyle='--', linewidth=1, label='Oversold (30)')
axes[2].set_title('RSI (14)')
axes[2].set_ylim(0, 100)
axes[2].legend(); axes[2].grid(alpha=0.3)

# MACD
macd_vals = data['MACD'].squeeze()
colors = ['green' if v >= 0 else 'red' for v in macd_vals]
axes[3].bar(data.index, macd_vals, color=colors, alpha=0.6, width=1)
axes[3].axhline(0, color='black', linewidth=0.8)
axes[3].set_title('MACD Histogram')
axes[3].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 5. Preprocessing & Scaling

**Key design decisions:**
- Each feature gets its **own scaler** — Close is ~$100–300, Volume is in millions, RSI is 0–100. Scaling them together would distort the model.
- Scaler is **fit only on training data** — no leakage into test set.

In [ ]:
feature_data = data[FEATURES].values   # shape: (N, 4)

split = int(len(feature_data) * TRAIN_SPLIT)
train_raw = feature_data[:split]
test_raw  = feature_data[split:]

print(f"Training rows : {len(train_raw)}")
print(f"Test rows     : {len(test_raw)}")

In [ ]:
# One scaler per feature — prevents large-magnitude features dominating
scalers = {}
train_scaled = np.zeros_like(train_raw, dtype=np.float32)
test_scaled  = np.zeros_like(test_raw,  dtype=np.float32)

for i, feat in enumerate(FEATURES):
    sc = MinMaxScaler(feature_range=(0, 1))
    train_scaled[:, i] = sc.fit_transform(train_raw[:, i].reshape(-1, 1)).flatten()
    test_scaled[:, i]  = sc.transform(test_raw[:, i].reshape(-1, 1)).flatten()
    scalers[feat] = sc

close_scaler = scalers['Close']   # kept for inverse-transforming predictions
print("Scaling complete. Feature ranges after scaling (should all be 0–1):")
for i, f in enumerate(FEATURES):
    print(f"  {f:8s} : [{train_scaled[:,i].min():.3f}, {train_scaled[:,i].max():.3f}]")

## 6. Sequence Building

Lookback window increased to **150 days** — gives the model more historical context per prediction, which helps with identifying longer-term trends and reversals.

In [ ]:
def build_sequences(scaled_data, lookback):
    """Build (X, y) sequences. X shape: (samples, lookback, features). y: Close only."""
    X, y = [], []
    for i in range(lookback, scaled_data.shape[0]):
        X.append(scaled_data[i - lookback:i, :])   # all features
        y.append(scaled_data[i, 0])                 # predict Close (index 0)
    return np.array(X, dtype=np.float32), np.array(y, dtype=np.float32)

x_train, y_train = build_sequences(train_scaled, LOOKBACK)

# Prepend last LOOKBACK rows of train to give test sequences their context
test_with_context = np.concatenate([train_scaled[-LOOKBACK:], test_scaled], axis=0)
x_test,  y_test  = build_sequences(test_with_context, LOOKBACK)

print(f"x_train : {x_train.shape}   y_train : {y_train.shape}")
print(f"x_test  : {x_test.shape}    y_test  : {y_test.shape}")

## 7. Model Architecture

Same 4-layer stacked LSTM. Input shape is now `(150, 4)` instead of `(100, 1)` — the model receives 4 feature channels per timestep.

In [ ]:
model = Sequential([
    Input(shape=(x_train.shape[1], x_train.shape[2])),

    LSTM(units=50, activation='relu', return_sequences=True),
    Dropout(0.2),

    LSTM(units=60, activation='relu', return_sequences=True),
    Dropout(0.3),

    LSTM(units=80, activation='relu', return_sequences=True),
    Dropout(0.4),

    LSTM(units=120, activation='relu'),
    Dropout(0.5),

    Dense(units=1)
])

model.compile(optimizer='adam', loss='mean_squared_error')
model.summary()

## 8. Training

In [ ]:
callbacks = [
    EarlyStopping(
        monitor='val_loss',
        patience=7,              # Slightly more patient than v1 (5→7)
        restore_best_weights=True,
        verbose=1
    ),
    ModelCheckpoint(
        filepath=MODEL_PATH,
        monitor='val_loss',
        save_best_only=True,
        verbose=1
    )
]

history = model.fit(
    x_train, y_train,
    epochs=100,                  # Higher ceiling — EarlyStopping decides the real stop
    batch_size=32,
    validation_split=0.1,
    callbacks=callbacks,
    verbose=1
)

In [ ]:
plt.figure(figsize=(12, 4))
plt.plot(history.history['loss'],     label='Training Loss',   color='blue',   linewidth=1.5)
plt.plot(history.history['val_loss'], label='Validation Loss', color='orange', linewidth=1.5)
best_epoch = np.argmin(history.history['val_loss'])
plt.axvline(best_epoch, color='red', linestyle='--', linewidth=1,
            label=f'Best epoch ({best_epoch})')
plt.title('Model Loss Over Epochs')
plt.xlabel('Epoch')
plt.ylabel('MSE Loss')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()
print(f"Best epoch      : {best_epoch}")
print(f"Best val_loss   : {min(history.history['val_loss']):.6f}")

## 9. Evaluation

In [ ]:
y_predict = model.predict(x_test, verbose=0).flatten()

# Inverse-transform Close price only
y_predict_real = close_scaler.inverse_transform(y_predict.reshape(-1, 1)).flatten()
y_actual_real  = close_scaler.inverse_transform(y_test.reshape(-1, 1)).flatten()

# Align dates with test predictions (first test prediction = row at index `split`)
test_dates = data.index[split : split + len(y_actual_real)]

In [ ]:
mae  = mean_absolute_error(y_actual_real, y_predict_real)
rmse = np.sqrt(mean_squared_error(y_actual_real, y_predict_real))
mape = np.mean(np.abs((y_actual_real - y_predict_real) / y_actual_real)) * 100

print("═" * 40)
print(f"  Stock : {STOCK} ({CURRENCY})")
print(f"  MAE   : {CURRENCY} {mae:.4f}")
print(f"  RMSE  : {CURRENCY} {rmse:.4f}")
print(f"  MAPE  : {mape:.2f}%")
print("═" * 40)

In [ ]:
n = min(len(y_actual_real), len(y_predict_real), len(test_dates))

plt.figure(figsize=(14, 6))
plt.plot(test_dates[:n], y_actual_real[:n],
         color='green', linewidth=1.5, label='Actual Price')
plt.plot(test_dates[:n], y_predict_real[:n],
         color='red',   linewidth=1.5, linestyle='--', label='Predicted Price')

# Confidence band ±3%
plt.fill_between(test_dates[:n],
                 y_predict_real[:n] * 0.97,
                 y_predict_real[:n] * 1.03,
                 color='red', alpha=0.1, label='±3% Band')

plt.title(f'{STOCK} — Actual vs Predicted Close Price (Multivariate LSTM)')
plt.xlabel('Date')
plt.ylabel(f'Price ({CURRENCY})')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Prediction error over time — useful to spot where the model struggles most
errors = y_actual_real[:n] - y_predict_real[:n]

plt.figure(figsize=(14, 3))
plt.bar(test_dates[:n], errors,
        color=['green' if e >= 0 else 'red' for e in errors],
        alpha=0.6, width=1)
plt.axhline(0, color='black', linewidth=0.8)
plt.title('Prediction Error (Actual − Predicted)')
plt.xlabel('Date')
plt.ylabel(f'Error ({CURRENCY})')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 10. Save Model

`ModelCheckpoint` already saved the best model during training to `Stock Predictions Model.keras`. The cell below saves the final epoch as a backup.

In [ ]:
model.save('Stock Predictions Model_final.keras')
print(f"✅ Best model  → '{MODEL_PATH}'")
print(f"✅ Final model → 'Stock Predictions Model_final.keras'")
print(f"   Input shape : (batch, {LOOKBACK}, {len(FEATURES)})")
print(f"   Features    : {FEATURES}")